In [ ]:
# ---------------------------------------------------------------------------
# Segmentation overlay generator
# ---------------------------------------------------------------------------
# This script performs a voxel-wise comparison between two binary segmentation
# volumes — a pipeline-generated segmentation and a reference segmentation —
# and produces a color-coded overlay that visually highlights areas of
# agreement and disagreement. True positives (TP, voxels detected by both
# segmentations) are shown in white, false positives (FP, detected by the pipeline
# but absent in the reference) in red, and false negatives (FN, missed by the
# pipeline but present in the reference) in blue. The overlay is saved as a
# single multi-slice RGB TIFF stack for inspection in any volume viewer, and a 
# subset of slices is displayed inline for a quick visual quality check.
# ---------------------------------------------------------------------------

import os
import numpy as np
import imageio
import tifffile
import matplotlib.pyplot as plt
from natsort import natsorted

# ---------------------------------------------------------------------------
# Configuration — replace these values before running
# ---------------------------------------------------------------------------
# Absolute path to the folder containing the pipeline-generated segmentation
# slices (one 2D TIFF per slice, grayscale, any non-zero value = foreground).
segmentation_folder = "/path/to/your/segmentation"

# Absolute path to the folder containing the Ground Truth reference segmentation slices
# (same format and slice count as the pipeline segmentation above).
reference_folder = "/path/to/reference/segmentation"

# Absolute path (including filename) for the saved RGB overlay TIFF stack.
output_overlay = "/path/to/output/overlay_stack_rgb.tif"

# ---------------------------------------------------------------------------
# Helper function — folder to 3D array
# ---------------------------------------------------------------------------
def load_tif_stack(folder_path):
    # natsorted ensures slices are loaded in natural numerical order
    # (e.g. slice_2.tif before slice_10.tif) rather than lexicographic order.
    file_list = natsorted([f for f in os.listdir(folder_path) if f.endswith('.tif')])
    stack = [imageio.imread(os.path.join(folder_path, f)) for f in file_list]
    # np.stack combines the list of 2D arrays into a single 3D array (Z, Y, X).
    return np.stack(stack, axis=0)

# ---------------------------------------------------------------------------
# Load segmentation volumes
# ---------------------------------------------------------------------------
your_segmentation = load_tif_stack(segmentation_folder)
reference_segmentation = load_tif_stack(reference_folder)

# Verify that both volumes have identical dimensions before comparison.
# A shape mismatch indicates a preprocessing error and must be resolved first.
assert your_segmentation.shape == reference_segmentation.shape, \
    "Shape mismatch between segmentations — check that both folders contain the same number of slices."

print(f"Pipeline segmentation shape : {your_segmentation.shape}")
print(f"Reference segmentation shape: {reference_segmentation.shape}")

# ---------------------------------------------------------------------------
# Binarisation
# ---------------------------------------------------------------------------
# Convert both volumes to binary masks (0 = background, 1 = foreground).
# Any non-zero intensity is treated as a foreground detection, making the
# script compatible with both uint8 (0/255) and labelled masks.
your_seg = (your_segmentation > 0).astype(np.uint8)
ref_seg  = (reference_segmentation > 0).astype(np.uint8)

# ---------------------------------------------------------------------------
# Colour-coded overlay construction
# ---------------------------------------------------------------------------
# For each slice, a three-channel (RGB) image is built by classifying every
# voxel into one of three categories:
#   True  Positive (TP) — detected by both              → white  [255, 255, 255]
#   False Positive (FP) — detected by pipeline only     → red    [255,   0,   0]
#   False Negative (FN) — present in reference only     → blue   [  0,   0, 255]
#   True  Negative (TN) — absent in both                → black  [  0,   0,   0]
overlay_stack = []

for i in range(your_seg.shape[0]):

    your_slice = your_seg[i]
    ref_slice  = ref_seg[i]

    overlay = np.zeros((your_slice.shape[0], your_slice.shape[1], 3), dtype=np.uint8)

    tp = (your_slice == 1) & (ref_slice == 1)
    fp = (your_slice == 1) & (ref_slice == 0)
    fn = (your_slice == 0) & (ref_slice == 1)

    overlay[tp] = [255, 255, 255]   # white  — TP
    overlay[fp] = [255,   0,   0]   # red    — FP
    overlay[fn] = [  0,   0, 255]   # blue   — FN

    overlay_stack.append(overlay)

# ---------------------------------------------------------------------------
# Inline preview — sample of slices
# ---------------------------------------------------------------------------
# Display up to 10 evenly spaced slices for a quick visual quality check.
for i in range(0, len(overlay_stack), max(len(overlay_stack) // 10, 1)):
    plt.figure(figsize=(5, 5))
    plt.imshow(overlay_stack[i])
    plt.title(f"Overlay — Slice {i}")
    plt.axis('off')
    plt.show()

# ---------------------------------------------------------------------------
# Save overlay stack
# ---------------------------------------------------------------------------
# Stack the list of 2D RGB arrays into a single 4D array (Z, Y, X, 3) and
# save as a multi-page RGB TIFF for inspection in ImageJ or
# any other volume viewer that supports colour TIFF stacks.
overlay_np = np.stack(overlay_stack, axis=0)
tifffile.imwrite(output_overlay, overlay_np)

print(f"Done: overlay stack saved to '{output_overlay}'.")